# Coding practice: build a BPE tokenizer

You will write the byte pair encoding loop from lecture 1.1, then the encoder that applies what it learns to new
text. At the end you will find a word where the order of the merges changes the answer.

> Save your own copy first: File → Save a copy in Drive.

In [ ]:
import re

CORPUS = "the reader read a thread . readers read threads . she reads the red thread and rereads it ."
PIECE = re.compile(r" ?[a-z]+| ?[^\sa-z]+|\s+")

def pretokenize(text):
    """Split text into words, each keeping the space before it, and punctuation. Merges never cross these pieces."""
    return PIECE.findall(text.lower())

print(pretokenize(CORPUS)[:8])

## 1. Count neighbouring pairs

Every piece starts as a list of single characters. `count_pairs` takes those lists and returns how often each
pair of neighbouring tokens occurs, counting inside pieces only.

<details><summary>Hint</summary>

Look at positions `i` and `i + 1` of each list. Add each pair to the dictionary the first time you meet it and
increase its count after that. Python dictionaries remember that order, which matters in part 3.

</details>

In [ ]:
def count_pairs(pieces):
    """pieces: a list of lists of tokens. Return a dict mapping (left, right) to how often that pair occurs."""
    counts = {}
    # TODO: count every pair of neighbouring tokens inside each piece.
    raise NotImplementedError("Count the pairs.")
    return counts

In [ ]:
toy = [list("abab"), list(" ab")]
got = count_pairs(toy)
assert got == {("a", "b"): 3, ("b", "a"): 1, (" ", "a"): 1}, got
assert list(got)[0] == ("a", "b"), "Add pairs in the order you first meet them."
print("count_pairs works.")

## 2. Merge one pair

`merge_pair` replaces every occurrence of a pair in one piece with the joined token, scanning from left to right.

<details><summary>Hint</summary>

Walk an index along the list. When the token there and the next one form the pair, append their concatenation
and step past both; otherwise append the token and step past one.

</details>

In [ ]:
def merge_pair(piece, pair):
    """Return a new list with every occurrence of `pair` joined into one token, scanning left to right."""
    # TODO: build and return the merged list.
    raise NotImplementedError("Merge the pair.")

In [ ]:
assert merge_pair(list("abab"), ("a", "b")) == ["ab", "ab"]
assert merge_pair(list("aaa"), ("a", "a")) == ["aa", "a"]
assert merge_pair(["t", "he", " ", "t", "he"], ("t", "he")) == ["the", " ", "the"]
print("merge_pair works.")

## 3. Learn the merges

`train_bpe` starts every piece of the corpus as single characters and repeats three steps: count the pairs, merge
the most frequent pair everywhere, and record it. It stops after `num_merges` merges, or sooner once no pair
occurs twice, because merging a pair seen once saves nothing.

Predict: after 12 merges, will every word in the corpus be a single token?

<details><summary>Hint</summary>

`max(counts, key=counts.get)` gives the most frequent pair, and on a tie the one counted first. Merge it in
every piece before you count again.

</details>

In [ ]:
def train_bpe(text, num_merges):
    """Return (merges, pieces): the merges in the order learned, and the corpus pieces after the last one."""
    pieces = [list(p) for p in pretokenize(text)]
    merges = []
    # TODO: up to num_merges times, count the pairs, take the most frequent, stop if it occurs fewer than
    # twice, merge it in every piece, and record it.
    raise NotImplementedError("Write the training loop.")
    return merges, pieces

In [ ]:
merges, pieces = train_bpe(CORPUS, 12)
tokens = [t for p in pieces for t in p]
assert "".join(tokens) == CORPUS, "Merging must never lose or add a character."
assert len(merges) == 12, len(merges)
lengths = [sum(len(p) for p in train_bpe(CORPUS, k)[1]) for k in range(13)]
assert all(b < a for a, b in zip(lengths, lengths[1:])), "Every merge should make the corpus shorter."
print("Merges, in order:", ["".join(m) for m in merges])
print("Corpus length after 0 to 12 merges:", lengths)
print("The corpus now:", " | ".join(tokens))

## 4. Encode new text

`encode` pre-tokenizes new text, starts each piece from single characters, and applies the learned merges one at a
time, in the order they were learned.

<details><summary>Hint</summary>

Loop over the merges in order, and for each one apply `merge_pair` to every piece.

</details>

In [ ]:
def encode(text, merges):
    """Return the tokens for `text`, applying `merges` in the order given."""
    # TODO: pre-tokenize, split each piece into characters, then apply every merge in order.
    raise NotImplementedError("Write the encoder.")

In [ ]:
assert encode(CORPUS, merges) == tokens, "Encoding the training text should give back the split training ended with."
for text in ["the thread", "readers reread", "a red hat ."]:
    assert "".join(encode(text, merges)) == text, text
print(encode("the rethreader reads", merges))

## 5. Does the order matter?

A shortcut people try is to skip the merge list and, at each position, take the longest token in the vocabulary
that fits. The next cell provides that encoder so you can compare the two.

Predict: will the shortcut and your `encode` split "the rear thread" the same way?

In [ ]:
def encode_longest_first(text, merges):
    vocab = set(CORPUS) | {left + right for left, right in merges}
    out = []
    for piece in pretokenize(text):
        i = 0
        while i < len(piece):
            j = max(j for j in range(i + 1, len(piece) + 1) if piece[i:j] in vocab or j == i + 1)
            out.append(piece[i:j])
            i = j
    return out

text = "the rear thread"
print("in order:     ", encode(text, merges))
print("longest first:", encode_longest_first(text, merges))

## Interpret what you found

1. Compare how the two encoders split " rear". In yours, which merge used up the `r` and `e` before `' re'` could
   use them, and how late in the list was `' re'` learned?
2. Encoding the training text gave back exactly the split training ended with, but the shortcut need not. Why?
3. "rethreader" never appears in the corpus. How many tokens does your encoder make of it, and what would a
   word-level vocabulary built from this corpus do with it?

Lab 1 builds on a tokenizer like this one.